In [0]:
import json
import os

import mlflow
from dotenv import load_dotenv


# Set up Databricks or local MLflow tracking
def is_databricks() -> bool:
    """Check if the code is running in a Databricks environment."""
    return "DATABRICKS_RUNTIME_VERSION" in os.environ

In [0]:
if not is_databricks():
    load_dotenv()
    profile = os.environ.get("PROFILE")
    mlflow.set_tracking_uri(f"databricks://{profile}")
    mlflow.set_registry_uri(f"databricks-uc://{profile}")

mlflow.get_tracking_uri()

In [0]:
experiment = mlflow.set_experiment(experiment_name="/Shared/telco_churn")
mlflow.set_experiment_tags({"repository_name": "mlops/telco_churn"})

print(experiment)

In [0]:
# dump class attributes in a json file for visualization
os.makedirs("../demo_artifacts", exist_ok=True)
with open("../demo_artifacts/mlflow_experiment.json", "w") as json_file:
    json.dump(experiment.__dict__, json_file, indent=4)

In [0]:
# get experiment by id
mlflow.get_experiment(experiment.experiment_id)

In [0]:
# search for experiment
experiments = mlflow.search_experiments(
    filter_string="tags.repository_name='mlops/telco_churn'"
)
print(experiments)

In [0]:
# get active run
print(mlflow.active_run().__dict__)

In [0]:
mlflow.end_run()
print(mlflow.active_run() is None)

In [0]:
# start a run
with mlflow.start_run(
    run_name="telco_churn-run",
    tags={"git_sha": "1234567890abcd"},
    description="telco churn prediction demo run",
) as run:
    run_id = run.info.run_id
    mlflow.log_params({"type": "churn_demo"})
    mlflow.log_metrics({"metric1": 1.0, "metric2": 2.0})

In [0]:
run_info = mlflow.get_run(run_id=run_id).to_dictionary()
print(run_info)

In [0]:
with open("../demo_artifacts/run_info.json", "w") as json_file:
    json.dump(run_info, json_file, indent=4)

In [0]:
print(run_info["data"]["metrics"])
print(run_info["data"]["params"])

In [0]:
mlflow.end_run()
print(mlflow.active_run() is None)

In [0]:
with mlflow.start_run(
    run_name="telco_churn_demo2",
    tags={"git_sha": "12345383hfyt",
          "test1": "tested"},
    description="teste de experiment tracking para telco churn",
) as run:
    run_id = run.info.run_id
    mlflow.log_params({"max_depth": 16,
                       "type": "telco_churn_type"})
    mlflow.log_metrics({"RMSE": 104.45})

print(mlflow.active_run() is None)

In [0]:
run_info = mlflow.get_run(run_id=run_id).to_dictionary()
print(run_info)

with open("../demo_artifacts/run_info.json", "w") as json_file:
    json.dump(run_info, json_file, indent=4)

In [0]:
print(run_info["data"]["metrics"])
print(run_info["data"]["params"])

In [0]:
run_id = mlflow.search_runs(filter_string="tags.git_sha='12345383hfyt'").run_id[0]
mlflow.end_run()
mlflow.start_run(run_id=run_id)
mlflow.log_artifact("/Workspace/Users/rafaelmp.cwb@gmail.com/mlops_custumerChurn/notebooks/catboost_info/time_left.tsv")

In [0]:
mlflow.end_run()

In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

client.log_dict(
    run_id=run_id,
    dictionary={"Rafael": "Dias"}, 
    artifact_file="dict_example.json"


)

import matplotlib.pyplot as plt

fig, ax = plt.subplots()
ax.plot([0, 1], [2, 3])

client.log_figure(
    run_id=run_id,
    figure=fig,
    artifact_file= "figure.png")